# Clase Teórica: Técnicas de Búsqueda (Unidad 2)
**Materia:** Desarrollo de Sistemas de Inteligencia Artificial.

**Unidad 2:**  Técnicas de búsqueda.

**Profesor:** Federico Balbuena  

## 1. Formulación de Problemas de Búsqueda

### 1.1 ¿Qué es un Problema de Búsqueda?
En Inteligencia Artificial, un agente orientado a metas necesita encontrar una **secuencia de acciones** que lo lleve desde una situación inicial hasta un estado deseado (meta).

* **Ejemplo cotidiano:** Un GPS calculando la mejor ruta entre dos ciudades, o un bot resolviendo el cubo de Rubik.

### 1.2 Componentes Formales de un Problema
Para formalizar un problema de búsqueda necesitamos 5 elementos:
1. **Estado Inicial:** El punto de partida del agente.
2. **Acciones Posibles:** Operaciones válidas aplicables a un estado $s$ 	→ $A(s)$.
3. **Modelo de Transición (Función Resultado):** Define qué estado se alcanza al ejecutar una acción $a$ en el estado $s$	→ $RESULT(s, a)$.
4. **Prueba de Objetivo (Goal Test):** Condición para determinar si un estado es la meta.
5. **Costo de Camino:** Valor numérico que suma el costo de cada paso realizado.

---  
### 1.3 Concepto Clave: Estado vs. Nodo
* **Estado:** Representa la configuración física o abstracta del mundo (ej. "estar en la ciudad B"). No tiene historia.
* **Nodo:** Es la estructura de datos que usa el algoritmo para construir el árbol de búsqueda. Contiene:
  * El `estado` actual.
  * El `nodo padre` (de dónde vino).
  * La `acción` que lo generó.
  * El `costo acumulado g(n)` desde el origen.

---  
### 1.4 ¿Qué es un Grafo y la Frontera?
* **Grafo:** Es una red compuesta por **vértices (nodos/estados)** conectados por **aristas (transiciones/acciones)** que pueden tener un peso o costo.
* **Frontera (Open List):** La lista de nodos que han sido generados pero aún no han sido explorados. La **estructura de datos** que elijamos para la frontera (Cola FIFO, Pila LIFO, Cola de Prioridad) define qué algoritmo de búsqueda estamos ejecutando.
* **Conjunto Explorado (Closed List):** Registro de estados ya visitados para evitar ciclos infinitos.

### 1.5 Representación del Grafo en Código
Explicación para la clase: Vamos a representar las ciudades como letras y los caminos entre ellas con sus correspondientes costos en kilómetros (o minutos). Por ejemplo, ir de 'A' a 'B' cuesta 2 unidades.

In [2]:
# Representación de un Grafo mediante un Diccionario de Adyacencia en Python
# Formato: 'Origen': [('Destino', Costo)]
grafo = {
    'A': [('B', 2), ('C', 1)],
    'B': [('G', 2)],
    'C': [('D', 1)],
    'D': [('G', 5)],
    'G': []  # Estado Objetivo (Goal)
}

print("Grafo cargado correctamente.")
print("Conexiones desde A:", grafo['A'])

Grafo cargado correctamente.
Conexiones desde A: [('B', 2), ('C', 1)]



## 2. Búsquedas No Informadas (Ciegas) : 
Estas estrategias **no poseen información sobre qué tan cerca está un estado de la meta**. Solo saben expandir nodos y verificar si llegaron al objetivo.

---  
### 2.1 Búsqueda en Anchura (BFS - Breadth-First Search)
* **Idea:** Explora el grafo nivel por nivel. Primero todos los vecinos a distancia 1, luego a distancia 2, etc.
* **Estructura de la Frontera:** **Cola FIFO** (First In, First Out).
* **Propiedad:** Garantiza encontrar el camino con el **menor número de pasos**, pero ignora los costos de las aristas.

```text
Grafo de Ejemplo:
       (2)           (2)
    [A] ----> [B] --------> ((G)) Meta
     |                       ^
    (1)                     (5)
     v                       |
    [C] ----> [D] -----------+
       (1)           
```

| Paso | Extraído (Salió) | Vecinos Descubiertos | Cola FIFO / Frontera (Siguiente a salir a la izq.) | Visitados / Explorados |
| :---: | :---: | :---: | :---: | :---: |
| **0** | — | — | `[A]` | `{A}` |
| **1** | **A** | B, C | `[B, C]` | `{A, B, C}` |
| **2** | **B** | G | `[C, G]` | `{A, B, C, G}` |
| **3** | **C** | D | `[G, D]` | `{A, B, C, G, D}` |
| **4** | **G** | **¡ES META!** | *(Se detiene)* | `{A, B, C, G, D}` |

In [ ]:
from collections import deque
#"Usamos deque porque en BFS necesitamos sacar elementos siempre por el principio (FIFO).
# Con una lista normal (pop(0)), Python tendría que reordenar toda la memoria en cada paso.
# Con deque.popleft(), la extracción toma tiempo constante y el algoritmo corre a máxima velocidad."

def bfs(grafo, inicio, objetivo):
    # -------------------------------------------------------------------------
    # 1. INICIALIZACIÓN DE ESTRUCTURAS DE DATOS
    # -------------------------------------------------------------------------
    # La FRONTERA es una Cola FIFO (First In, First Out).
    # Guardamos tuplas con dos datos: (estado_actual, camino_recorrido_hasta_aqui).
    # Usamos 'deque' para que extraer por la izquierda (popleft) tome tiempo constante O(1).
    frontera = deque([(inicio, [inicio])])
    
    # Conjunto de VISITADOS (Explorados + Encolados).
    # Usamos un 'set' (conjunto) porque buscar un elemento ('x in visitados') 
    # toma tiempo O(1), a diferencia de una lista que tomaría O(n).
    # Agregamos 'inicio' de entrada para no volver a pasarlo a la frontera.
    visitados = {inicio}
    
    print(f"=== Inicio BFS: Buscando ruta de {inicio} a {objetivo} ===")
    paso = 0
    
    # -------------------------------------------------------------------------
    # 2. BUCLE PRINCIPAL DE BÚSQUEDA
    # Mientras existan nodos esperando en la frontera para ser evaluados:
    # -------------------------------------------------------------------------
    while frontera:
        paso += 1
        
        # PASO A: Extraer el nodo más antiguo en la frontera (Comportamiento FIFO / Anchura)
        estado, camino = frontera.popleft() #
        
        # Mostramos en consola el nodo que SALE de la cola y lo que QUEDA adentro esperando
        print(f"Paso {paso} | Evaluando Estado: {estado} | Frontera actual: {[e[0] for e in frontera]}")
        
        # PASO B: PRUEBA DE OBJETIVO (Ocurre formalmente al EXTRAER el nodo)
        if estado == objetivo:
            print(f"--> Meta encontrada en {paso} pasos!")
            return camino  # Retorna la lista de nodos del camino óptimo
            
        # PASO C: EXPANSIÓN DE VECINOS
        # Obtenemos las aristas conectadas al estado actual (grafo.get evita KeyErrors si no tiene salidas)
        for vecino, _ in grafo.get(estado, []):
            
            # Filtramos: Solo procesamos vecinos que NUNCA hayan entrado a la frontera ni hayan sido explorados
            if vecino not in visitados:
                
                # OPTIMIZACIÓN: Se marca como 'visitado' inmediatamente al descubrirlo.
                # Esto evita encolar duplicados si otro nodo del mismo nivel apunta al mismo vecino.
                visitados.add(vecino)
                
                # ENCOLAR: Se agrega al FINAL de la cola con su camino extendido
                frontera.append((vecino, camino + [vecino]))

    # -------------------------------------------------------------------------
    # 3. FRACASO DE BÚSQUEDA
    # Si la frontera se vacía y nunca se llegó al objetivo, no hay camino posible.
    # -------------------------------------------------------------------------
    return None

# Ejemplo de ejecución:
camino_bfs = bfs(grafo, 'A', 'G')
print("Camino hallado por BFS:", camino_bfs)

=== Inicio BFS: Buscando ruta de A a G ===
Paso 1 | Evaluando Estado: A | Frontera actual: []
Paso 2 | Evaluando Estado: B | Frontera actual: ['C']
Paso 3 | Evaluando Estado: C | Frontera actual: ['G']
Paso 4 | Evaluando Estado: G | Frontera actual: ['D']
--> Meta encontrada en 4 pasos!
Camino hallado por BFS: ['A', 'B', 'G']


### 2.2 Búsqueda de Costo Uniforme (UCS - Uniform Cost Search)
* **Idea:** En lugar de evaluar por nivel de profundidad, evalúa siempre el nodo con el **menor costo acumulado $g(n)$**.
* **Estructura de la Frontera:** **Cola de Prioridad / Min-Heap** (ordenada por $g(n)$).
* **Regla Fundamental:** La **prueba de objetivo** debe realizarse estrictamente al **extraer** el nodo de la frontera, NO al generarlo. Si la hacemos al generar, podríamos detenernos con un camino subóptimo que aún no contemplaba una ruta alternativa más barata.

In [ ]:
import heapq
# ¿Por qué heapq? heapq convierte una lista normal de Python en un Binary Min-Heap (Montículo Mínimo).
# Esto permite extraer siempre el costo mínimo (heappop) e insertar nuevos nodos (heappush)
# Tuplas en el Heap: Al guardar (costo, estado, camino), heapq usa el primer valor (costo) para decidir la prioridad. 
# Por eso el costo va sí o sí al principio.
# Lazy Deletion (Borrado Perezoso): En lugar de buscar y eliminar rutas desactualizadas dentro del heap (operación costosa), 
# se permite que convivan en la frontera y simplemente se descartan cuando salen usando if costo > mejor_costo.get(estado, float('inf'))

def costo_uniforme(grafo, inicio, objetivo):
    # -------------------------------------------------------------------------
    # 1. INICIALIZACIÓN DE ESTRUCTURAS DE DATOS
    # -------------------------------------------------------------------------
    # La FRONTERA es una Cola de Prioridad (Priority Queue).
    # Usamos una lista común de Python que luego manipularemos con la librería 'heapq'.
    # Guardamos: (costo_acumulado_g, estado_actual, camino_recorrido)
    frontera = [(0, inicio, [inicio])]
    
    # Diccionario 'mejor_costo' (g_cost):
    # Almacena el menor costo conocido hasta el momento para llegar a cada nodo.
    # Nos permite saber si encontramos un camino más corto hacia un nodo ya visto.
    mejor_costo = {inicio: 0}
    
    print(f"=== Inicio UCS (Costo Uniforme): {inicio} -> {objetivo} ===")
    paso = 0
    
    # -------------------------------------------------------------------------
    # 2. BUCLE PRINCIPAL DE BÚSQUEDA
    # -------------------------------------------------------------------------
    while frontera:
        paso += 1
        
        # PASO A: Extraer de la frontera el nodo con el MENOR costo acumulado g(n)
        # heapq.heappop remueve y devuelve siempre el mínimo elemento del Min-Heap en O(log N).
        costo, estado, camino = heapq.heappop(frontera)
        
        # Visualización en consola de lo que se extrae y qué queda esperando en la frontera
        print(f"Paso {paso} | Extraído: {estado} (g={costo}) | Frontera: {[(e[1], 'g=' + str(e[0])) for e in frontera]}")
        
        # PASO B: Control de Obsolescencia ("Lazy Deletion")
        # Si sacamos un nodo pero su costo es MAYOR que el mejor costo que ya registramos
        # para ese mismo estado, significa que es una ruta vieja/peor que quedó colgada.
        # Simplemente la descartamos y pasamos a la siguiente iteración.
        if costo > mejor_costo.get(estado, float('inf')):
            continue
            
        # PASO C: PRUEBA DE OBJETIVO (Ocurre estrictamente al EXTRAER)
        # UCS garantiza optimalidad justamente porque evaluamos la meta al salir de la frontera
        # (cuando se garantiza que ya no hay ninguna ruta con un costo menor).
        if estado == objetivo:
            print(f"--> Meta alcanzada con costo óptimo g = {costo}")
            return camino, costo
            
        # PASO D: EXPANSIÓN Y EVALUACIÓN DE VECINOS
        # Obtenemos los vecinos y el costo individual del paso hacia cada uno: (vecino, costo_paso)
        for vecino, costo_paso in grafo.get(estado, []):
            
            # Calculamos el costo acumulado real g(n) desde el inicio hasta el vecino
            nuevo_costo = costo + costo_paso
            
            # Si encontramos un camino nuevo hacia 'vecino' que es MÁS BARATO que el anterior:
            # inf: Representa un costo infinitamente alto. Se usa como valor por defecto si es la primera vez que el programa ve a ese vecino
            if nuevo_costo < mejor_costo.get(vecino, float('inf')):
                # 1. Actualizamos la tabla del mejor costo
                mejor_costo[vecino] = nuevo_costo
                # 2. Insertamos en el Min-Heap en tiempo O(log N)
                heapq.heappush(frontera, (nuevo_costo, vecino, camino + [vecino]))
                
    # -------------------------------------------------------------------------
    # 3. FRACASO
    # -------------------------------------------------------------------------
    return None
'''
grafo = {
    'A': [('B', 2), ('C', 1)],
    'B': [('G', 2)],
    'C': [('D', 1)],
    'D': [('G', 5)],
    'G': []  # Estado Objetivo (Goal)
}
'''
# Ejemplo de ejecución:
camino_ucs, costo_total = costo_uniforme(grafo, 'A', 'G')
print("Camino hallado por UCS:", camino_ucs, "con Costo:", costo_total)

=== Inicio UCS (Costo Uniforme): A -> G ===
Paso 1 | Extraído: A (g=0) | Frontera: []
Paso 2 | Extraído: C (g=1) | Frontera: [('B', 'g=2')]
Paso 3 | Extraído: B (g=2) | Frontera: [('D', 'g=2')]
Paso 4 | Extraído: D (g=2) | Frontera: [('G', 'g=4')]
Paso 5 | Extraído: G (g=4) | Frontera: []
--> Meta alcanzada con costo óptimo g = 4
Camino hallado por UCS: ['A', 'B', 'G'] con Costo: 4


## 3. Búsqueda Informada: Heurísticas y Algoritmo A*

### 3.1 ¿Qué es una Heurística $h(n)$?
Es una estimación del costo restante desde un nodo $n$ hasta el objetivo. Es información propia del dominio del problema que "guía" la búsqueda para evitar explorar ramas innecesarias.

* **Admisibilidad:** Una heurística es admisible si **nunca sobreestima** el costo real para alcanzar la meta ($0 \le h(n) \le h^*(n)$). Esta propiedad garantiza que A* encuentre la solución óptima.

---

#### Ejemplos Geométricos Frecuentes (Entornos en Grilla / Mapas)
Cuando el espacio de estados tiene coordenadas $(x, y)$, las heurísticas más utilizadas son:

1. **Distancia Manhattan ($L_1$):**  
   Se utiliza cuando el agente solo puede moverse en 4 direcciones (arriba, abajo, izquierda, derecha), sin movimientos diagonales. Suma las diferencias absolutas en cada eje:
   $$h(n) = |x_1 - x_2| + |y_1 - y_2|$$

2. **Distancia Euclídea ($L_2$):**  
   Mide la distancia "en línea recta" entre dos puntos (la hipotenusa del triángulo entre ambos estados):
   $$h(n) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

> **Nota:** Hoy usamos un grafo abstracto con un diccionario de valores $h(n)$ fijos. En la clase práctica del laboratorio aplicaremos estas fórmulas geométricas sobre un mapa en grilla (*Grid World*).


**Con dibujitos**

```text
(x1, y1)  o-------------------------> Cateto Horizontal (|x1 - x2|)
          |                      .
          |                   .  
 Cateto   |                .     
Vertical  |             .         <-- Hipotenusa / Euclídea (Línea recta)
(|y1-y2|) |          .        
          |       .           
          v    .              
       (x2, y2) Meta

* Manhattan (Recorrido en "L") = Cateto Horizontal + Cateto Vertical
* Euclídea  (Línea recta)     = Hipotenusa
```
---

### 3.2 El Algoritmo A*
A* combina lo mejor de dos mundos:
* $g(n)$: El costo real acumulado desde el inicio hasta el nodo actual (mira hacia el pasado).
* $h(n)$: La estimación heurística hasta la meta (mira hacia el futuro).

$$\text{Función de Evaluación: } f(n) = g(n) + h(n)$$

A* prioriza la expansión de los nodos con el menor valor de $f(n)$. Si $h(n)$ es admisible, A* garantiza encontrar el camino de menor costo.

In [ ]:
import heapq

'''grafo = {
    'A': [('B', 2), ('C', 1)],
    'B': [('G', 2)],
    'C': [('D', 1)],
    'D': [('G', 5)],
    'G': []  # Estado Objetivo (Goal)
}'''

# Tabla de heurísticas h(n) estimadas hacia la meta 'G'
heuristica = {
    'A': 3,
    'B': 2,
    'C': 4,
    'D': 5,
    'G': 0
}

def a_estrella(grafo, inicio, objetivo, h):
    # -------------------------------------------------------------------------
    # 1. INICIALIZACIÓN DE ESTRUCTURAS DE DATOS
    # -------------------------------------------------------------------------
    # En A*: f(n) = g(n) + h(n)
    # En el nodo inicial: g = 0, por lo que f_inicial = 0 + h[inicio]
    f_inicial = 0 + h[inicio]
    
    # La FRONTERA es una Cola de Prioridad (Min-Heap).
    # Como 'heapq' ordena por el primer término, ponemos a f_inicial en primer lugar.
    # Estrategia del orden en la tupla: (f(n), g(n), estado_actual, camino_recorrido)
    frontera = [(f_inicial, 0, inicio, [inicio])]
    
    # Guarda la menor distancia real conocida g(n) desde 'inicio' hasta cada nodo.
    mejor_g = {inicio: 0}
    
    print(f"=== Inicio Algoritmo A*: {inicio} -> {objetivo} ===")
    paso = 0
    
    # -------------------------------------------------------------------------
    # 2. BUCLE PRINCIPAL DE BÚSQUEDA
    # -------------------------------------------------------------------------
    while frontera:
        paso += 1
        
        # PASO A: Extraer el nodo con el menor costo estimado f(n) = g(n) + h(n)
        f, g, estado, camino = heapq.heappop(frontera)
        
        # Visualización en consola
        print(f"Paso {paso} | Extraído: {estado} (f={f}, g={g}, h={h[estado]}) | Frontera: {[(e[2], 'f=' + str(e[0])) for e in frontera]}")
        
        # PASO B: Control de Obsolescencia ("Lazy Deletion")
        # Si sacamos un nodo pero su g real es peor que el 'mejor_g' registrado, se descarta.
        if g > mejor_g.get(estado, float('inf')):
            continue
            
        # PASO C: PRUEBA DE OBJETIVO AL EXTRAER
        # Al igual que en UCS, A* garantiza optimalidad si la prueba de objetivo
        # se realiza al EXTRAER de la frontera (y la heurística es admisible).
        if estado == objetivo:
            print(f"--> Solución óptima A* encontrada en el paso {paso}!")
            return camino, g
            
        # PASO D: EXPANSIÓN Y EVALUACIÓN DE VECINOS
        for vecino, costo_paso in grafo.get(estado, []):
            
            # Calculamos el costo acumulado real g(n) para el vecino
            nuevo_g = g + costo_paso
            
            # Si encontramos una ruta real MÁS BARATA hacia el vecino:
            if nuevo_g < mejor_g.get(vecino, float('inf')):
                # 1. Actualizamos su mejor g(n)
                mejor_g[vecino] = nuevo_g
                
                # 2. Calculamos el valor de evaluación f(n) = g(n) + h(n)
                nuevo_f = nuevo_g + h[vecino]
                
                # 3. Insertamos el nodo en la cola de prioridad
                heapq.heappush(frontera, (nuevo_f, nuevo_g, vecino, camino + [vecino]))
                
    # -------------------------------------------------------------------------
    # 3. FRACASO
    # -------------------------------------------------------------------------
    return None

# Ejemplo de ejecución:
camino_a, costo_a = a_estrella(grafo, 'A', 'G', heuristica)
print("Camino hallado por A*:", camino_a, "con Costo Total:", costo_a)

=== Inicio Algoritmo A*: A -> G ===
Paso 1 | Extraído: A (f=3, g=0, h=3) | Frontera: []
Paso 2 | Extraído: B (f=4, g=2, h=2) | Frontera: [('C', 'f=5')]
Paso 3 | Extraído: G (f=4, g=4, h=0) | Frontera: [('C', 'f=5')]
--> Solución óptima A* encontrada en el paso 3!
Camino hallado por A*: ['A', 'B', 'G'] con Costo Total: 4


## 4. Cuadro Comparativo y Errores Frecuentes

| Algoritmo | Criterio de Selección | ¿Es Completo? | ¿Es Óptimo? | Estructura de Frontera |
| :--- | :--- | :---: | :---: | :--- |
| **BFS** | El nodo más somero (menos pasos) | Sí | Sí (si todos los costos son iguales) | Cola FIFO (`deque`) |
| **DFS** | El nodo más profundo | No (en grafos infinitos) | No | Pila LIFO (`list.append/pop`) |
| **UCS** | Menor costo acumulado $g(n)$ | Sí | Sí | Cola de Prioridad (`heapq`) |
| **A*** | Menor $f(n) = g(n) + h(n)$ | Sí | Sí (si $h(n)$ es admisible) | Cola de Prioridad (`heapq`) |

---  
### Errores Frecuentes en Exámenes e Implementación:
1. **Evaluar la meta al ENCOLAR en UCS/A*:** Lleva a aceptar caminos que no son de costo mínimo.
2. **No usar la lista de explorados/visitados:** Genera bucles infinitos en grafos con ciclos.
3. **Heurísticas no admisibles:** Sobreestimar el costo futuro rompe la garantía de encontrar la solución óptima en A*.